# Monday Morning at NorthStar

> *Sarah Chen · Customer Experience Analyst · NorthStar Retail · her first week.*

**Estimated time: 10–15 minutes. Run every cell top to bottom.**

This is your first hands-on notebook. Run it **before class** — the goal is for you to *feel* what ML can do first, and *understand why* later.

## The Scene

It is 9:12 AM on a Monday — the first week of Sarah's story. You are **Sarah Chen**, starting your second week as Customer Experience Analyst at NorthStar Retail — a mid-sized online clothing store.

**Aisha Patel** from Customer Service walks into your cube holding a USB drive:

> *"Priya wants sentiment on every customer review we got in December. All ten thousand of them. By Friday. Apparently you're the new analyst so — good luck."*

She drops the drive on your desk and leaves.

You open the CSV. 10,000 rows. Reading one review takes about 15 seconds. That's **42 hours of reading** — more than a work-week — before you write a single line of analysis.

You could try a shortcut: write `if-else` rules that look for positive and negative words. Or you could try this "Machine Learning" thing you've been hearing about.

Let's try both.

## Step 1 — Set up

We'll use a pre-trained sentiment model called **DistilBERT** from Hugging Face Transformers. It was trained on millions of labelled reviews and classifies text as positive or negative with high accuracy — no rules to write.

**First run:** the model (~250 MB) downloads and caches automatically. After that, it runs offline.

In [1]:
import os
# Must be set before importing transformers to prevent tokenizer deadlock on macOS
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd                  # pandas = tool for working with tables of data
from transformers import pipeline    # pipeline = a ready-made, pre-trained AI model we can just use
import warnings                      # lets us hide harmless warning messages

# macOS guard: cap PyTorch threads to 1 to prevent the OpenMP scheduler from hanging.
import torch
# (Removed the old one-CPU-thread cap — it made CPU-only machines much slower.)

warnings.filterwarnings("ignore")    # hide harmless warnings so the output stays clean

print("✅ Libraries ready.")

✅ Libraries ready.


## Step 2 — The reviews

Here are 5 real-looking reviews from NorthStar's December haul. Read them carefully before running the next cell — try to guess each one's sentiment in your head.

In [2]:
# Five real-looking customer reviews — a tiny sample of Sarah's 10,000
reviews = [
    "Absolutely love this jacket! Fits perfectly and the fabric feels premium. Will buy again.",
    "The shirt arrived two weeks late and the color is nothing like the photo. Very disappointed.",
    "Great value for the price. Not amazing but not bad — I'd order again for basics.",
    "I ordered size M as usual and it's enormous. Runs at least two sizes large. Returning.",
    "Sure, it's 'fine', if you enjoy paying $85 for something that comes apart after one wash.",
]

# Print each review with a number in front (enumerate counts them for us, starting at 1)
for i, r in enumerate(reviews, 1):
    print(f'{i}. {r}\n')

1. Absolutely love this jacket! Fits perfectly and the fabric feels premium. Will buy again.

2. The shirt arrived two weeks late and the color is nothing like the photo. Very disappointed.

3. Great value for the price. Not amazing but not bad — I'd order again for basics.

4. I ordered size M as usual and it's enormous. Runs at least two sizes large. Returning.

5. Sure, it's 'fine', if you enjoy paying $85 for something that comes apart after one wash.



### Pause & Predict

Before running the next cell, predict in your head:
- Which of the 5 reviews is **positive**?
- Which is **negative**?
- Is any one of them **tricky**?

Write your predictions down (even just mentally) before scrolling.

## Step 3 — Attempt 1: hand-written rules

Let's try the traditional approach. We'll write a rule: if a review contains more positive words than negative words, label it positive.

This is how programmers solved problems before ML. Watch what happens.

In [3]:
# Sarah's first attempt: hand-written word lists — "if it says 'love', it's positive"
POSITIVE_WORDS = ['love', 'great', 'amazing', 'good', 'perfect', 'premium']
NEGATIVE_WORDS = ['bad', 'disappointed', 'late', 'enormous', 'terrible', 'worst']

def rule_based_sentiment(text):
    text_lower = text.lower()                                  # lowercase so 'Love' matches 'love'
    pos = sum(1 for w in POSITIVE_WORDS if w in text_lower)    # count positive words found
    neg = sum(1 for w in NEGATIVE_WORDS if w in text_lower)    # count negative words found
    if pos > neg:                                              # more positive words -> call it positive
        return 'positive'
    elif neg > pos:                                            # more negative words -> call it negative
        return 'negative'
    else:                                                      # tie (or no hits) -> we can't tell
        return 'neutral / unclear'

# Run the rules on all 5 reviews and print the verdicts
for i, r in enumerate(reviews, 1):
    label = rule_based_sentiment(r)
    print(f'{i}. [{label:17}]  {r[:70]}...')

1. [positive         ]  Absolutely love this jacket! Fits perfectly and the fabric feels premi...
2. [negative         ]  The shirt arrived two weeks late and the color is nothing like the pho...
3. [positive         ]  Great value for the price. Not amazing but not bad — I'd order again f...
4. [negative         ]  I ordered size M as usual and it's enormous. Runs at least two sizes l...
5. [neutral / unclear]  Sure, it's 'fine', if you enjoy paying $85 for something that comes ap...


### What do you notice?

Look carefully at what the rule-based classifier labelled each review. Compare to your own predictions.

- Review 3 ("Great value... not amazing but not bad") — does the rule handle mixed opinions well?
- Review 5 ("Sure, it's 'fine', if you enjoy paying $85...") — this is sarcasm. Did the rule catch it?
- What other words would you need to add to the lists? When would this ever end?

**This is the core problem the rule-based approach was never going to solve.** You cannot write a finite rulebook for language. People are too creative.

## Step 4 — Attempt 2: Machine Learning

Now let's try ML. We'll use a **DistilBERT** model from Hugging Face that was pre-trained on millions of labelled reviews. It learned which words, phrases, and *context* tend to appear in positive vs. negative text — we did not write those rules ourselves.

We load that model and use it with one line.

**No hand-written rules. One line.**

In [4]:
# Now the ML way: use a model someone else already trained on millions of examples
print("Loading pre-trained sentiment model... (first run may take ~1 min)")
_pipeline = pipeline("sentiment-analysis")   # this one line loads the whole trained model
print("✅ Model ready.")

def ml_sentiment(text):
    result = _pipeline(text)[0]              # ask the model for its verdict on one review
    label = result["label"].lower()          # "positive" or "negative"
    confidence = result["score"]             # how sure the model is (0 to 1)
    return label, confidence

# Print a header row, then the model's verdict on each of the 5 reviews
print(f"{'#':3} {'label':10} {'confidence':>10}  review")
print('-' * 90)
for i, r in enumerate(reviews, 1):
    label, confidence = ml_sentiment(r)
    print(f'{i:<3} {label:10} {confidence:>10.2%}  {r[:60]}...')

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading pre-trained sentiment model... (first run may take ~1 min)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Model ready.
#   label      confidence  review
------------------------------------------------------------------------------------------
1   positive       99.99%  Absolutely love this jacket! Fits perfectly and the fabric f...
2   negative       99.97%  The shirt arrived two weeks late and the color is nothing li...
3   positive       99.63%  Great value for the price. Not amazing but not bad — I'd ord...
4   positive       96.80%  I ordered size M as usual and it's enormous. Runs at least t...
5   positive       99.93%  Sure, it's 'fine', if you enjoy paying $85 for something tha...


### What do you notice?

Compare the ML output to your own predictions, and to the rule-based classifier.

- The **confidence score** (between 0% and 100%) tells you *how sure* the classifier is — not just *which label* it picked. Lower confidence = a signal to route a case to a human reviewer.
- **Look closely at review 5** (the sarcastic one: *"Sure, it's 'fine', if you enjoy paying $85..."*). What did the model label it? Is that right?
- **Look at review 4** (the sizing complaint). The word 'enormous' isn't negative by itself — the model may have been unsure.
- Notice that you did not have to teach the model about *every* word. It generalised from the labelled examples it saw during training.

**Key insight:** The ML model is better than the rules *on average*, but it **still makes mistakes** — especially on sarcasm, negation, and context-heavy text. It is a pre-trained model, not magic. Keep that in mind for the next step.

## Step 5 — The punch line

Now let's run the ML model at scale — not 5 reviews, but thousands. To keep the wait short on any laptop we'll run a **2,000-review slice** of Sarah's 10,000 (the timing lesson is identical; if your machine is fast, change `2_000` to `10_000` in the next cell and see the full workload).

In [5]:
# Simulate 10k reviews by repeating and varying our 5 seeds
import random          # random = tool for making random choices
random.seed(42)        # fix the randomness so everyone gets the same result

# Small endings we can tack onto reviews to make them vary a little
variants = [
    '',
    ' Arrived in good condition.',
    ' Shipping was fine.',
    ' Would recommend to a friend.',
    ' Not what I expected.',
    ' Thanks for the fast delivery!',
    ' The packaging could be better.',
]

# Build 2,000 reviews: each one = a random seed review + a random ending
# (2,000 keeps the wait short on slower laptops — bump to 10_000 for Sarah's full workload)
big_batch = [random.choice(reviews) + random.choice(variants) for _ in range(2_000)]
print(f'Total reviews to classify: {len(big_batch):,}')

Total reviews to classify: 10,000


In [6]:
import time                                        # lets us measure how long things take

start = time.time()                                # note the clock before we start
labels = [ml_sentiment(r)[0] for r in big_batch]   # classify the whole batch, keep just the label
elapsed = time.time() - start                      # how many seconds did that take?

print(f'Classified {len(big_batch):,} reviews in {elapsed:.1f} seconds.')
print()

# Count how many reviews got each label (positive vs negative)
result = pd.Series(labels).value_counts()
print('Breakdown:')
print(result.to_string())

Classified 10,000 reviews in 106.5 seconds.

Breakdown:
positive    6876
negative    3124


### What do you notice?

- **Manual effort for this task:** ~42 hours of reading.
- **ML effort for this task:** what you just saw — a few seconds.
- Sarah's Friday deadline just became trivial. The rest of the week she can spend on *what this report means*, not on reading.

This is why ML matters in business: **scale that humans cannot match, on problems where a human expert would do it if only they had the time.**

## Step 6 — Sarah's moment of doubt

Sarah writes up her findings and walks into Priya's office with the numbers she just saw.

> *Sarah: "Roughly 60% of December reviews are positive, 20% negative, 20% neutral."*

Priya reads the numbers. Then looks up:

> *Priya: "Sarah — are the positive ones **actually** positive? And how do we know the model isn't miscounting? You mentioned earlier it got a sarcastic review wrong. How many more of those are in the 10,000?"*

Sarah opens her mouth and closes it again. She doesn't know.

She can see the model ran. She can see the numbers. But she has no way to *prove* the numbers are right. The sarcastic review from Step 4 is just one error she caught by hand. How many silent errors are hiding in the other 9,995 reviews?

That question — *how sure are we?* — is the single most important question in applied ML. It is the question Lesson 2 will teach you to answer. And Lessons 3 and 4. And every lesson after that, in some form.

**For now, bring Priya's question to class.** The instructor will help you start to answer it.

## ✅ End of Notebook

You have just completed the first step of your pre-class work.

**Next step →** Go back to [`pre-class.md`](../pre-class.md) and continue with Step 2 (Reflect). The whole pre-class cycle takes ~30 minutes. You've used about 15.

When you get to class, Part 1 will revisit what you just saw — but with a real, heavyweight model (Hugging Face transformers) and a harder set of reviews.